In [ ]:
import pandas as pd
from src.train import *
from src.model import CNNLSTMModel
from src.dataset import *
from src.preprocess import *
from src.visualize import *
import torch
import json

## Hyper-Parameter

In [ ]:
performe_visualization = False

In [ ]:
# some important parameters
pre_day = 5

feature_cols = [
    "metric.STATUS_AC_MOD_ADMISSION_TEMP.MEASURED",  # ambient temperature
    "metric.STATUS_INTERNAL_TEMP.MEASURED",          # internal temperature
    "metric.AC_VOLTAGE_AB.MEASURED",                 # AC voltage
    "metric.AC_VOLTAGE_BC.MEASURED",                 # AC voltage
    "metric.AC_VOLTAGE_CA.MEASURED",                 # AC voltage
    "metric.DC_VOLTAGE.MEASURED",                  # DC voltage
    "metric.AC_POWER.MEASURED",                     # AC power
]

exclude_periods = [
    [pd.Timestamp('2021-01-01'), pd.Timestamp('2021-12-23')], # data collection issue
    [pd.Timestamp('2023-02-23'), pd.Timestamp('2023-08-26')], # anomalies in the data
]

## Data Pipeline


### Load Data

In [ ]:
inverter_data = load_parquet_data('dataset/inverter_data')
failure_sessions = load_failure_sessions('dataset/failure_sessions_w_maintenance.csv', min_days=3)

In [ ]:
visualize_failure_timeline(failure_sessions)

In [ ]:
if performe_visualization:
    # Visualize the raw data
    visualize_mean_values(inverter_data, failure_sessions, feature_cols, 'visualization', 'raw_data')

In [ ]:
# remove unused columns
filtered_data = inverter_data[['event_local_time', 'device_name'] + feature_cols].copy()

### Anomoly Detection

In [ ]:
if "metric.STATUS_AC_MOD_ADMISSION_TEMP.MEASURED" in filtered_data.columns:
    anomaly_ids = filtered_data["metric.STATUS_AC_MOD_ADMISSION_TEMP.MEASURED"]>=100
    filtered_data.loc[anomaly_ids, "metric.STATUS_AC_MOD_ADMISSION_TEMP.MEASURED"] = None
    print("Anomalies in STATUS_AC_MOD_ADMISSION_TEMP removed:", anomaly_ids.sum())

### Missing Value Imputation

In [ ]:
imputed_df = missing_value_imputation(
    filtered_data, feature_cols, 
    time_col='event_local_time', 
    device_col='device_name', 
    short_gap_limit=0, 
    long_fill_value=0.0, 
    add_missing_mask=True
    )
extended_feature_cols = feature_cols+[col+'_missing' for col in feature_cols]

### Downsampling

In [ ]:
downsampled_data = downsample_inverter_raw(imputed_df, drop_empty_bins=False)
downsampled_data.dropna(inplace=True) # NaN will be generated by downsampling, so we drop them

In [ ]:
if performe_visualization:
    # Visualize the downsampled data
    visualize_mean_values(
        downsampled_data, 
        failure_sessions, 
        extended_feature_cols, 
        'visualization', 
        'downsampled_data',
        freq=None
    )

### Data Cleaning

In [ ]:
print("failure_sessions shape:", failure_sessions.shape)
failure_sessions['event_local_time'] = failure_sessions['start_time']
filtered_sessions = exclude_periods_from_data(failure_sessions, exclude_periods)
filtered_sessions['event_local_time'] = filtered_sessions['end_time']
filtered_sessions = exclude_periods_from_data(filtered_sessions, exclude_periods)
print("failure_sessions shape:", filtered_sessions .shape)

In [ ]:
visualize_failure_timeline(filtered_sessions)

In [ ]:
print("inverter_data shape:", downsampled_data.shape)
downsampled_data = exclude_periods_from_data(downsampled_data, exclude_periods)
print("Excluded data shape:", downsampled_data.shape)

### Data Labeling

In [ ]:
downsampled_data = prepare_dataset(downsampled_data, failure_sessions, pre_days=pre_day)

### Feature Engineering

In [ ]:
# month_of_year range 1~12
downsampled_data['month_sin'] = np.sin(2 * np.pi * downsampled_data['event_local_time'].dt.month / 12)
downsampled_data['month_cos'] = np.cos(2 * np.pi * downsampled_data['event_local_time'].dt.month / 12)

# If there's also hour_of_day (0~23), can be converted similarly
downsampled_data['hour_sin'] = np.sin(2 * np.pi * downsampled_data['event_local_time'].dt.hour / 24)
downsampled_data['hour_cos'] = np.cos(2 * np.pi * downsampled_data['event_local_time'].dt.hour / 24)

extended_feature_cols += ['hour_sin', 'hour_cos', 'month_sin', 'month_cos']

In [ ]:
v = downsampled_data[['metric.AC_VOLTAGE_AB.MEASURED','metric.AC_VOLTAGE_BC.MEASURED','metric.AC_VOLTAGE_CA.MEASURED']]
v_mean = v.mean(axis=1)
v_range = v.max(axis=1) - v.min(axis=1)
downsampled_data['V_mean'] = v_mean
downsampled_data['V_unbalance'] = v_range / (v_mean + 1e-6)

extended_feature_cols += ['V_mean', 'V_unbalance']

In [ ]:
downsampled_data

In [ ]:
downsampled_data['T_delta'] = downsampled_data['metric.STATUS_INTERNAL_TEMP.MEASURED'] - downsampled_data['metric.STATUS_AC_MOD_ADMISSION_TEMP.MEASURED']
extended_feature_cols += ['T_delta']

In [ ]:
if performe_visualization:
    visualize_mean_values(downsampled_data, failure_sessions, extended_feature_cols+['label'], 
                                title='processed_data', freq=None)

### Split Dataset

In [ ]:
split_time = [pd.Timestamp('2024-06-30'), pd.Timestamp('2025-01-01')]


train_df = downsampled_data[downsampled_data['event_local_time'] <= split_time[0]].copy()
val_df = downsampled_data[(downsampled_data['event_local_time'] > split_time[0]) & (downsampled_data['event_local_time'] <= split_time[1])].copy()
test_df = downsampled_data[downsampled_data['event_local_time'] > split_time[1]].copy()

In [ ]:
print('train set period:', train_df['event_local_time'].min(), train_df['event_local_time'].max())
print('validation set period:', val_df['event_local_time'].min(), val_df['event_local_time'].max())
print('test set period:', test_df['event_local_time'].min(), test_df['event_local_time'].max())

In [ ]:
extended_feature_cols

### Standardization

In [ ]:
from scipy.stats import f_oneway

def anova_test(df, feature, device_col='device_name'):
    groups = [df[df[device_col] == d][feature].dropna() 
              for d in df[device_col].unique()]
    stat, p = f_oneway(*groups)
    return stat, p
anova_results = {col: anova_test(train_df, col) for col in extended_feature_cols}

In [ ]:
anova_results_df = pd.DataFrame(anova_results, index=['F-statistic', 'p-value']).T
anova_results_df.sort_values(by='p-value', inplace=True)
anova_results_df

In [ ]:
import pandas as pd

def check_device_wise_stats(df, feature_cols, device_col='device_name'):
    stats = df.groupby(device_col)[feature_cols].agg(['mean', 'std'])
    return stats

check_device_wise_stats(downsampled_data, extended_feature_cols, device_col='device_name')

In [ ]:
from sklearn.preprocessing import StandardScaler
scalers = {}
feature_to_standardize = feature_cols + ['V_mean', 'V_unbalance', 'T_delta']

for device in train_df['device_name'].unique():
    device_data = train_df[train_df['device_name'] == device].copy()
    scaler = StandardScaler()
    scalers[device] = scaler
    
    device_data[extended_feature_cols] = scaler.fit_transform(device_data[extended_feature_cols])
    train_df.loc[device_data.index, extended_feature_cols] = device_data[extended_feature_cols]
    val_df.loc[val_df['device_name'] == device, extended_feature_cols] = scaler.transform(val_df[val_df['device_name'] == device][extended_feature_cols])
    test_df.loc[test_df['device_name'] == device, extended_feature_cols] = scaler.transform(test_df[test_df['device_name'] == device][extended_feature_cols])


In [ ]:
if performe_visualization:
    visualize_mean_values(train_df, failure_sessions, extended_feature_cols+['label'], 
                                 'visualization', 'train_data', freq=None)
    visualize_mean_values(val_df, failure_sessions, extended_feature_cols+['label'], 
                                 'visualization', 'val_data', freq=None)
    visualize_mean_values(test_df, failure_sessions, extended_feature_cols+['label'],
                                    'visualization', 'test_data', freq=None)

## Save Dataset

In [ ]:
train_df.to_csv('dataset/train_data.csv', index=False)
val_df.to_csv('dataset/val_data.csv', index=False)
test_df.to_csv('dataset/test_data.csv', index=False)

In [ ]:
dataset_parameters = {
    "feature_cols": extended_feature_cols,
}
with open("config/dataset_parameters.json", "w") as f:
    json.dump(dataset_parameters, f, indent=4)
extended_feature_cols